# 30-Day Mortality Prediction after Acute Myocardial Infarction
## Full analysis — GitHub portfolio edition

**Author:** Tanjim Hossain  
**Environment used for the submitted analysis:** Python 3.12.13 / Kaggle / scikit-learn 1.6.1

This notebook is the GitHub-facing companion to the fully executed Kaggle analysis supplied with the project. The numerical outputs below are the preserved results of that executed analysis; the reusable implementation lives in `../src/`. The portfolio edition removes environment-specific clutter while retaining the complete analytical narrative, model-selection logic, outputs and interpretation.

## Executive summary

- 785 patients, 52 deaths (6.62%), 17 candidate predictors.
- Leakage-safe preprocessing: all imputation, scaling and encoding fitted inside training folds.
- Repeated nested validation: 5-fold outer CV × 5 repeats; 4-fold inner CV tuned by log loss.
- Elastic-net development strongly favoured the ridge/L2 endpoint.
- Final model: ridge logistic regression, `C=0.1`, no class rebalancing.
- ROC-AUC 0.7881, PR-AUC 0.2783, Brier 0.0556, log loss 0.2070.
- Calibration intercept 0.1277, slope 1.0580.
- Internal validation only; no clinical deployment claim.

## 1. Setup and reproducibility

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../src').resolve()))
import isds_option_a_pipeline as isds

RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)
DATA = Path('../ami_patient_data.csv')
print('Python 3.12.13 | numpy 2.0.2 | pandas 2.3.3 | matplotlib 3.10.0 | seaborn 0.13.2 | scikit-learn 1.6.1')
print(f'Random seed: {RANDOM_SEED}')

Python 3.12.13 | numpy 2.0.2 | pandas 2.3.3 | matplotlib 3.10.0 | seaborn 0.13.2 | scikit-learn 1.6.1
Random seed: 2026


## 2. Raw-data audit and deterministic cleaning

In [2]:
raw, data = isds.load_and_clean(DATA)
audit = isds.audit_table(raw, data)
display(audit)
display(data.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('missing_cells'))

Rows: 785 | Columns: 18 | Candidate predictors: 17
Deaths: 52 | Event rate: 6.62%
Explicit missing cells: 7
Hypothension encoded as Unknown: 3
Killip_class encoded as -1: 14
Height outside stated range [140, 212]: 2
Exact duplicate rows: 0
Duplicate predictor profiles: 0


Cleaning rules used in the original analysis:

| Raw issue | Treatment |
|---|---|
| `Hypothension` | rename to `Hypotension` |
| `Hyperthension` | rename to `Hypertension` |
| `Hypothension=Unknown` | recode as missing |
| `Killip_class=-1` | recode as missing |
| `Height=1.75` | correct to 175 cm |
| `Height=1690` | correct to 169 cm |

No patient was deleted. After recoding there were 24 missing predictor cells: Killip class 15, hypotension 4, gender 2, and one each for age, anterior infarct location and height.

![Outcome imbalance and missingness](../docs/assets/data_audit.svg)

## 3. Leakage-safe preprocessing

In [3]:
X = data.drop(columns=[isds.OUTCOME]).copy()
y = data[isds.OUTCOME].astype(int).copy()
preprocessor = isds.make_preprocessor()
print(f'Patients: {len(X)}')
print(f'Candidate predictors: {X.shape[1]}')
print(f'Deaths: {int(y.sum())}')
print(f'Event rate: {y.mean():.2%}')
print('No preprocessing has been fitted to the complete dataset.')

Patients: 785
Candidate predictors: 17
Deaths: 52
Event rate: 6.62%
No preprocessing has been fitted to the complete dataset.


Primary preprocessing is fitted **inside training folds only**: median imputation + standardisation for continuous/ordinal predictors, most-frequent imputation for binary predictors, and most-frequent imputation + one-hot encoding for smoking. Killip class is ordinal in the primary analysis.

## 4. Repeated nested cross-validation

In [4]:
OUTER_SPLITS, OUTER_REPEATS, INNER_SPLITS = 5, 5, 4
print('Outer CV: 5-fold stratified CV x 5 repeats')
print('Total outer test folds: 25')
print('Inner CV: 4-fold stratified CV')
print('Tuning criterion: log loss')

Outer CV: 5-fold stratified CV x 5 repeats
Total outer test folds: 25
Inner CV: 4-fold stratified CV
Tuning criterion: log loss


Each patient receives one genuine out-of-fold probability in each outer repeat. The five probabilities are averaged for headline patient-level estimates; repeat-level metrics are retained to assess stability.

## 5. Elastic-net development screen

In [5]:
# Full reproducible implementation: ../src/elastic_net_screen.py
elastic_grid = {
    'C': [0.01, 0.1, 1.0],
    'l1_ratio': [0.00, 0.25, 0.50, 0.75, 1.00],
}
display(pd.read_csv('../results/tables/elastic_net_selection_frequency.csv'))

Initial elastic-net OOF performance
ROC-AUC                       0.7878
PR-AUC                        0.2790
Brier score                   0.0556
Log loss                      0.2071

Selection frequency
C      l1_ratio  outer_folds_selected
0.1    0.00      20
1.0    0.75       2
0.1    0.25       1
1.0    0.00       1
1.0    1.00       1

Twenty of 25 outer folds selected `C=0.1` and `l1_ratio=0`, i.e. the ridge/L2 endpoint. This supported retaining all predictors with shrinkage rather than imposing a hard sparse subset.

## 6. Candidate-model comparison

In [6]:
candidate_results = pd.read_csv('../results/tables/candidate_model_performance.csv')
display(candidate_results)

Model                     ROC-AUC  PR-AUC  Brier  Log loss
Ridge logistic             0.7881  0.2783  0.0556  0.2070
Random Forest              0.7806  0.2462  0.0564  0.2094
Standard logistic          0.7726  0.2816  0.0558  0.2127
Gradient Boosting          0.7509  0.2379  0.0571  0.2160
Intercept-only reference   0.4638  0.0631  0.0619  0.2438

![Candidate model comparison](../docs/assets/model_comparison.svg)

Ridge was selected for the best overall balance of discrimination, probability accuracy, calibration and repeated-validation stability — not ROC-AUC alone.

## 7. Class-imbalance ablation

In [7]:
imbalance = pd.read_csv('../results/tables/class_imbalance_ablation.csv')
display(imbalance)

Strategy              ROC-AUC  PR-AUC  Brier  Log loss
No rebalancing         0.7881   0.2783  0.0556  0.2070
Class weighting        0.7746   0.2712  0.1711  0.5138
Random oversampling    0.7703   0.2700  0.1700  0.5108

Both rebalancing strategies substantially worsened Brier score and log loss, so the natural class distribution was retained. Random oversampling was restricted to training folds in the original experiment.

## 8. Repeated-CV stability

| Model | ROC-AUC mean ± SD | PR-AUC mean ± SD | Brier mean ± SD | Log loss mean ± SD |
|---|---:|---:|---:|---:|
| Ridge | 0.7831 ± 0.0056 | 0.2610 ± 0.0310 | 0.0559 ± 0.0009 | 0.2084 ± 0.0026 |
| Random Forest | 0.7694 ± 0.0101 | 0.2251 ± 0.0312 | 0.0569 ± 0.0011 | 0.2124 ± 0.0041 |
| Standard logistic | 0.7677 ± 0.0067 | 0.2708 ± 0.0440 | 0.0565 ± 0.0020 | 0.2160 ± 0.0047 |
| Gradient Boosting | 0.7351 ± 0.0152 | 0.2144 ± 0.0474 | 0.0581 ± 0.0020 | 0.2202 ± 0.0066 |

## 9. Calibration and bootstrap uncertainty

In [8]:
calibration = pd.read_csv('../results/tables/calibration_summary.csv')
display(calibration)

Ridge calibration
intercept              0.1277
slope                  1.0580
mean predicted risk    0.0663
observed event rate    0.0662

![Calibration summary](../docs/assets/calibration.svg)

In [9]:
bootstrap_ci = pd.read_csv('../results/tables/bootstrap_95ci.csv')
display(bootstrap_ci)

Metric       Estimate  CI 2.5%  CI 97.5%
ROC-AUC       0.7881    0.7236    0.8478
PR-AUC        0.2783    0.1741    0.4054
Brier score   0.0556    0.0423    0.0690
Log loss      0.2070    0.1658    0.2487

The interval calculation used 2,000 patient-level bootstrap resamples of the final internally validated out-of-fold prediction set.

## 10. Sensitivity analyses

In [10]:
sensitivity = pd.read_csv('../results/tables/sensitivity_analyses.csv')
display(sensitivity)

Specification                 ROC-AUC  PR-AUC  Brier  Log loss
Primary ridge                  0.7881  0.2783  0.0556  0.2070
Killip categorical             0.7786  0.2233  0.0567  0.2103
Missing indicators added       0.7889  0.2787  0.0556  0.2069

The primary ordinal Killip representation was retained. Missingness indicators added essentially no practical improvement, so the simpler imputation-only specification was preferred.

## 11. Threshold trade-offs

In [11]:
thresholds = pd.read_csv('../results/tables/threshold_metrics.csv')
display(thresholds)

Threshold  Sensitivity  Specificity   PPV    NPV   Flagged
0.05        0.788        0.619       0.128  0.976   0.408
0.10        0.538        0.831       0.184  0.962   0.194
0.15        0.385        0.925       0.267  0.955   0.096
0.20        0.231        0.956       0.273  0.946   0.056

These thresholds are illustrative and were not optimised as treatment cut-offs on the development data.

## 12. Decision-curve analysis

In [12]:
dca = pd.read_csv('../results/tables/decision_curve_key_thresholds.csv')
display(dca)

Threshold  Ridge NB  Treat-all NB  Treat-none NB
0.05        0.0335      0.0171        0.0000
0.10        0.0181     -0.0375        0.0000
0.15        0.0131     -0.0985        0.0000
0.20        0.0051     -0.1672        0.0000

![Decision curve summary](../docs/assets/decision_curve.svg)

Within the original notebook's explored grid, ridge had greater estimated net benefit than both reference strategies over approximately 1%–30%. This is a prediction analysis; it does not establish causal treatment benefit.

## 13. Final ridge coefficients and odds ratios

In [13]:
coef = pd.read_csv('../results/tables/final_ridge_coefficients.csv')
display(coef)

Predictor                         Coefficient   Odds ratio
Age                                0.6554       1.9258
Heart_rate                         0.4691       1.5985
Time_To_Relief                     0.3208       1.3783
Previous_myocardial_infarction     0.3139       1.3687
Killip_class                       0.3072       1.3596
Anterior_infarct_location          0.2750       1.3165
Gender                            -0.2190       0.8033
Weight                            -0.2167       0.8052
Diabetes                           0.2082       1.2315
Hypotension                        0.1742       1.1903

Continuous/ordinal predictors were standardised, so their odds ratios correspond approximately to a one-standard-deviation increase. The coefficients describe predictive associations, not causal effects.

## 14. Outer-fold raw-predictor permutation importance

In [14]:
importance = pd.read_csv('../results/tables/permutation_importance.csv')
display(importance)

Predictor                         Mean log-loss increase
Age                                0.02714
Killip_class                       0.00881
Heart_rate                         0.00524
Time_To_Relief                     0.00268
Weight                             0.00259
Previous_myocardial_infarction     0.00171
Anterior_infarct_location          0.00134
Gender                             0.00100
Height                             0.00041
Hypotension                        0.00031

![Outer-fold permutation importance](../docs/assets/permutation_importance.svg)

## 15. Final conclusion

Ridge-penalised logistic regression without class rebalancing provided the strongest overall internally validated performance among the evaluated candidates. The model combined useful discrimination with good calibration and lower probability error than the rebalanced alternatives. Age was the strongest predictive contributor by held-out permutation analysis. The project is internally validated only; external validation is required before clinical use.

### Computational traceability
- Reusable full pipeline: `../src/isds_option_a_pipeline.py`
- Elastic-net development screen: `../src/elastic_net_screen.py`
- Machine-readable validated outputs: `../results/tables/`
- Methodology and responsible-use documentation: `../docs/`